<a href="https://colab.research.google.com/github/PriyanshuBhunia/classification-ML/blob/main/PCam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import f1_score
import kagglehub

# --- 1. Dataset Downloading & Path Setup ---
print("Downloading dataset...")
# Download the dataset
dataset_path = kagglehub.dataset_download("andrewmvd/metastatic-tissue-classification-patchcamelyon")

# Define paths to the specific H5 files
# Note: Adjusting paths based on standard structure; verify if your download differs.
img_path = os.path.join(dataset_path, "pcam", "training_split.h5")
lbl_path = os.path.join(dataset_path, "Labels", "Labels", "camelyonpatch_level_2_split_train_y.h5")

print(f"Image File: {img_path}")
print(f"Label File: {lbl_path}")

# --- 2. Custom Dataset Class ---
class PCamDataset(Dataset):
    def __init__(self, image_file, label_file, transform=None):
        self.image_file = image_file
        self.label_file = label_file
        self.transform = transform

        # We open files in __init__ for simplicity with num_workers=0
        # For multiple workers, opening in __getitem__ or using worker_init_fn is safer
        self.h5_x = h5py.File(image_file, 'r')['x']
        self.h5_y = h5py.File(label_file, 'r')['y']
        self.length = len(self.h5_x)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Read image and label
        image = self.h5_x[idx]
        # Extract scalar label explicitly to avoid DeprecationWarning
        # The labels are (N, 1, 1, 1), so we flatten and take the first element
        label = int(np.array(self.h5_y[idx]).flatten()[0])

        # Preprocess: Normalize 0-255 -> 0.0-1.0
        image = image.astype('float32') / 255.0

        if self.transform:
            image = self.transform(image)

        return image, label

# --- 3. Model Definition ---
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            # Input: 3 x 96 x 96
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # -> 32 x 48 x 48

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # -> 64 x 24 x 24

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)  # -> 128 x 12 x 12
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 12, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# --- 4. Main Training Loop ---
def main():
    # Device configuration
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device}")

    # Transformations
    # ToTensor converts numpy [0, 1] (H, W, C) -> torch tensor (C, H, W)
    data_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    # Initialize Dataset
    print("Initializing dataset...")
    full_dataset = PCamDataset(img_path, lbl_path, transform=data_transform)

    # Create a small subset (1000 samples)
    subset_indices = range(1000)
    subset_dataset = Subset(full_dataset, subset_indices)
    print(f"Created subset with {len(subset_dataset)} samples.")

    # DataLoader
    train_loader = DataLoader(subset_dataset, batch_size=32, shuffle=True, num_workers=0)

    # Initialize Model, Loss, and Optimizer
    model = SimpleCNN().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training
    epochs = 3
    print(f"\nStarting training for {epochs} epochs...")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        all_targets = []
        all_preds = []

        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.float().to(device).unsqueeze(1)

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            # Store predictions for metrics
            predicted = (outputs > 0.5).float()
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())

        # Calculate Epoch Metrics
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = np.mean(np.array(all_targets) == np.array(all_preds))
        epoch_f1 = f1_score(all_targets, all_preds)

        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | F1: {epoch_f1:.4f}")

    print("\nDone!")

if __name__ == "__main__":
    main()

Using Colab cache for faster access to the 'metastatic-tissue-classification-patchcamelyon' dataset.
Image File: /kaggle/input/metastatic-tissue-classification-patchcamelyon/pcam/training_split.h5
Label File: /kaggle/input/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_train_y.h5

Using device: cpu
Initializing dataset...
Created subset with 1000 samples.

Starting training for 3 epochs...
Epoch [1/3] | Loss: 0.6871 | Acc: 0.5590 | F1: 0.5353
Epoch [2/3] | Loss: 0.5667 | Acc: 0.7150 | F1: 0.7388
Epoch [3/3] | Loss: 0.5158 | Acc: 0.7730 | F1: 0.7824

Done!


mlp

In [54]:
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import f1_score
import kagglehub

# --- 1. Dataset Downloading & Path Setup ---
print("Downloading dataset...")
dataset_path = kagglehub.dataset_download("andrewmvd/metastatic-tissue-classification-patchcamelyon")

img_path = os.path.join(dataset_path, "pcam", "training_split.h5")
lbl_path = os.path.join(dataset_path, "Labels", "Labels", "camelyonpatch_level_2_split_train_y.h5")

# --- 2. Custom Dataset Class ---
class PCamDataset(Dataset):
    def __init__(self, image_file, label_file, transform=None):
        self.h5_x = h5py.File(image_file, 'r')['x']
        self.h5_y = h5py.File(label_file, 'r')['y']
        self.length = len(self.h5_x)
        self.transform = transform

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        image = self.h5_x[idx]
        # Flatten label array to extract scalar, fixing DeprecationWarning
        label = int(np.array(self.h5_y[idx]).flatten()[0])

        # Normalize to [0, 1]
        image = image.astype('float32') / 255.0

        if self.transform:
            image = self.transform(image)

        return image, label

# --- 3. MLP Model Definition ---
class PCamMLP(nn.Module):
    def __init__(self):
        super(PCamMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            # Input: 96*96*3 = 27648
            nn.Linear(96 * 96 * 3, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

# --- 4. Main Training Loop ---
def main():
    # Device configuration
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Transformations
    transform = transforms.Compose([
        transforms.ToTensor(), # Converts (H, W, C) -> (C, H, W)
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ])

    # Load Dataset
    print("Loading dataset...")
    full_dataset = PCamDataset(img_path, lbl_path, transform=transform)

    # Subset (1000 samples)
    subset = Subset(full_dataset, range(1000))
    loader = DataLoader(subset, batch_size=32, shuffle=True, num_workers=0)
    print(f"Subset created with {len(subset)} samples.")

    # Model, Loss, Optimizer
    model = PCamMLP().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training Loop
    print("Starting training for 3 epochs...")
    for epoch in range(3):
        model.train()
        epoch_loss = 0
        all_targets = []
        all_preds = []

        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.float().to(device).unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            # Metrics
            preds = (outputs > 0.5).float()
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

        # Calculate Epoch Stats
        avg_loss = epoch_loss / len(loader)
        acc = np.mean(np.array(all_targets) == np.array(all_preds))
        f1 = f1_score(all_targets, all_preds)

        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")

if __name__ == "__main__":
    main()

Using Colab cache for faster access to the 'metastatic-tissue-classification-patchcamelyon' dataset.
Using device: cpu
Loading dataset...
Subset created with 1000 samples.
Starting training for 3 epochs...
Epoch 1 | Loss: 0.9178 | Acc: 0.5750 | F1: 0.6126
Epoch 2 | Loss: 0.5798 | Acc: 0.7180 | F1: 0.7504
Epoch 3 | Loss: 0.4476 | Acc: 0.8100 | F1: 0.8291
